# VINH — TRAIN NLP SKILL EXTRACTION TRÊN VIETJOBS


**Nguồn:** [dinhieufam/VietJobs](https://huggingface.co/datasets/dinhieufam/VietJobs) — 48.092 tin tuyển dụng Việt Nam (toàn bộ lĩnh vực). Không yêu cầu API key; tải file CSV công khai (~103 MB).

**Mục tiêu:** Xây dựng **baseline phân loại kỹ năng đa nhãn** từ `job_title`, `description`, `requirements_text`; nhãn tham chiếu lấy từ `technical_skills`. Chỉ số F1 được tính trên **nhãn có sẵn/được chuẩn hóa của VietJobs**, **chưa chứng minh độ chính xác với nhãn người đánh giá**.

**Quy trình:** Tải dữ liệu → kiểm tra nhãn → lọc JD IT (có thể tắt) → chuẩn hóa các nhãn → chống trùng lặp → chia Train/Validation/Test → TF-IDF + Logistic Regression → tối ưu ngưỡng trên Validation → đánh giá Test → lưu model → (tùy chọn) ChromaDB/semantic retrieval.

**Lưu ý:** Bộ dữ liệu gốc chỉ có split `train`, notebook tự chia 3 tập. `technical_skills` có thể null hoặc là chuỗi list, cần xử lý trước khi train. Không được đưa `technical_skills` vào **input** mô hình vì sẽ gây *data leakage*.

## 1. Cài môi trường


In [1]:
import sys
import pandas as pd
import numpy as np

print(sys.version)
print(pd.__version__)
print(np.__version__)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
2.3.2
2.2.6


In [2]:
%pip install -q \
    "pandas==2.2.3" \
    "numpy==2.2.6" \
    "huggingface-hub>=0.25,<2.0" \
    scikit-learn \
    joblib \
    matplotlib

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.52.0 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import huggingface_hub
import sklearn

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("HuggingFace:", huggingface_hub.__version__)
print("Scikit-learn:", sklearn.__version__)

print("Các thư viện import thành công!")

Pandas: 2.2.3
NumPy: 2.4.6
HuggingFace: 1.33.0
Scikit-learn: 1.9.1
Các thư viện import thành công!


## 2. Import và cấu hình
- `ONLY_IT=True`: ưu tiên JD CNTT; có thể đổi thành `False` nếu muốn toàn bộ ngành nghề.
- `MAX_ROWS=12000`: giới hạn số JD **sau** khi lọc để chạy nhanh. Đặt `None` để sử dụng toàn bộ JD hợp lệ.
- `MIN_TRAIN_POS=12`: nhãn cần ít nhất 12 mẫu dương trong tập train.

In [5]:
import os, ast, re, json, unicodedata, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import joblib
from IPython.display import display
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, hamming_loss)

SEED = 42
ONLY_IT = True
MAX_ROWS = 12000            # None: toàn bộ JD hợp lệ
MIN_TRAIN_POS = 12          # số mẫu có kỹ năng đó trong tập TRAIN
MAX_LABELS = 35             # tối đa 35 lớp kỹ năng
MIN_TEXT_CHARS = 60
MODEL_DIR = Path('/content/vinh_vietjobs_models') if Path('/content').exists() else Path('vinh_vietjobs_models')
MODEL_DIR.mkdir(exist_ok=True, parents=True)
print('Output:', MODEL_DIR)

Output: /content/vinh_vietjobs_models


## 3. Tải dataset gốc từ Hugging Face (không API key)
Dataset ~103 MB. Nếu link lỗi vì mạng, chuyển sang ô *upload thủ công* phía dưới.

In [6]:
try:
    csv_path = hf_hub_download(
        repo_id='dinhieufam/VietJobs',
        filename='VietJobs.csv',
        repo_type='dataset'
    )
    print('Đã tải:', csv_path)
except Exception as error:
    print('Không tải tự động được:', str(error)[:250])
    print('Tải thủ công tại: https://huggingface.co/datasets/dinhieufam/VietJobs/blob/main/VietJobs.csv')
    print('Sau đó chạy ô dưới để upload CSV lên Colab.')
    csv_path = None

VietJobs.csv: reconstructing file:   0%|          |  0.00B /  103MB            

VietJobs.csv: downloading bytes:           |  0.00B            

Đã tải: /root/.cache/huggingface/hub/datasets--dinhieufam--VietJobs/snapshots/ea140511b77935704e93d21c2973b72f46d48902/VietJobs.csv


In [7]:
# CHỈ chạy ô này nếu csv_path là None, hoặc muốn dùng CSV tải thủ công.
if csv_path is None:
    from google.colab import files
    uploaded = files.upload()  # chọn VietJobs.csv
    csv_path = next(name for name in uploaded if name.lower().endswith('.csv'))

assert csv_path is not None, 'Chưa có VietJobs.csv'
df_raw = pd.read_csv(csv_path, low_memory=False, encoding='utf-8-sig')
print('Số JD ban đầu:', len(df_raw))
print('Cột:', df_raw.columns.tolist())
expected = {'job_title','description','requirements_text','technical_skills','category'}
assert expected.issubset(set(df_raw.columns)), f'Thiếu các cột: {expected - set(df_raw.columns)}'
display(df_raw[['job_title','category','technical_skills']].head(5))

Số JD ban đầu: 48092
Cột: ['job_title', 'location', 'country', 'qualifications', 'technical_skills', 'soft_skills', 'languages_required', 'experience_required', 'salary', 'contract_type', 'working_hours', 'benefits', 'description', 'requirements_text', 'category', 'salary_min', 'salary_max', 'salary_avg']


,job_title,category,technical_skills
0,Nhân Viên Vận Hành Hệ Thống Xử Lý Nước Thải,nông_nghiệp_năng_lượng_môi_trường,NaN
1,Nhân Viên Vận Hành Hệ Thống Xử Lí Nước Thải,nông_nghiệp_năng_lượng_môi_trường,"['Vận hành hệ thống xử lý nước thải', 'Phân tí..."
2,Nhân Viên Phát Triển Thị Trường,nông_nghiệp_năng_lượng_môi_trường,NaN
3,Nhân Viên Bảo Trì Hệ Thống Xử Lý Nước,nông_nghiệp_năng_lượng_môi_trường,NaN
4,Chuyên Gia Đánh Giá ISO 22000–HACCP–VIETGAP,nông_nghiệp_năng_lượng_môi_trường,"['HACCP', 'ISO 22000', 'VietGAP', 'GLOBAL GAP']"


## 4. Chuẩn hóa văn bản và kỹ năng
`technical_skills` là nhãn tham chiếu, thường lưu dạng chuỗi list Python. Để có các Skill ID nhất quán, mình gom các tên biến thể vào danh sách công nghệ phổ biến (Python, SQL, Docker, AWS...). Cách gom này có thể bỏ sót các kỹ năng ít gặp. Có thể thay bảng ánh xạ bằng taxonomy riêng của nhóm sau này.

In [8]:
SKILL_PATTERNS = {
    'Python':       [r'\bpython\b'],
    'Java':         [r'\bjava\b(?!\s*script)'],
    'JavaScript':   [r'\bjavascript\b', r'\bjava\s*script\b', r'\bjs\b'],
    'TypeScript':   [r'\btypescript\b'],
    'C#':           [r'(?<!\w)c\s*#(?!\w)', r'\bcsharp\b'],
    'C++':          [r'(?<!\w)c\s*\+\+(?!\w)'],
    'PHP':          [r'\bphp\b'],
    'Go':           [r'\bgolang\b', r'\bgo language\b'],
    'Kotlin':       [r'\bkotlin\b'],
    'Swift':        [r'\bswift\b'],
    'SQL':          [r'\bsql\b', r'\bsql server\b'],
    'MySQL':        [r'\bmysql\b'],
    'PostgreSQL':   [r'\bpostgres(?:ql)?\b'],
    'MongoDB':      [r'\bmongodb\b'],
    'Redis':        [r'\bredis\b'],
    'React':        [r'\breact(?:js|\.js)?\b'],
    'Angular':      [r'\bangular\b'],
    'Vue':          [r'\bvue(?:js|\.js)?\b'],
    'Node.js':      [r'\bnode(?:js|\.js)?\b'],
    'Spring Boot':  [r'\bspring\s*boot\b'],
    '.NET':         [r'(?<!\w)\.net\b', r'\basp\.?net\b'],
    'Django':       [r'\bdjango\b'],
    'FastAPI':      [r'\bfastapi\b'],
    'Laravel':      [r'\blaravel\b'],
    'HTML/CSS':     [r'\bhtml\b',r'\bcss\b'],
    'Docker':       [r'\bdocker\b'],
    'Kubernetes':   [r'\bkubernetes\b', r'\bk8s\b'],
    'Linux':        [r'\blinux\b'],
    'AWS':          [r'\baws\b', r'\bamazon web services\b'],
    'Azure':        [r'\bazure\b'],
    'GCP':          [r'\bgcp\b',r'\bgoogle cloud\b'],
    'Git':          [r'\bgit\b',r'\bgithub\b',r'\bgitlab\b'],
    'CI/CD':        [r'\bci\s*/\s*cd\b',r'\bcontinuous integration\b'],
    'TensorFlow':   [r'\btensorflow\b'],
    'PyTorch':      [r'\bpytorch\b'],
    'Machine Learning': [r'\bmachine learning\b', r'hoc may',r'\bml\b'],
    'NLP':          [r'\bnlp\b', r'xu ly ngon ngu tu nhien'],
    'Data Analysis':[r'\bdata analy(?:sis|tics)\b',r'phan tich du lieu'],
    'Power BI':     [r'\bpower\s*bi\b'],
    'Excel':        [r'\bexcel\b'],
    'REST API':     [r'\brest(?:ful)?\s*api\b',r'\bapi rest\b'],
    'Microservices': [r'\bmicroservices?\b', r'kien truc vi dich vu'],
    'Kafka':        [r'\bkafka\b'],
    'Spark':        [r'\bapache spark\b',r'\bspark\b'],
    'Selenium':     [r'\bselenium\b'],
    'Figma':        [r'\bfigma\b'],
    'Jira':         [r'\bjira\b'],
    'Scrum':        [r'\bscrum\b'],
}
COMPILED_SKILLS = {k: [re.compile(p, re.I) for p in patterns]
                   for k, patterns in SKILL_PATTERNS.items()}

def clean_text(value):
    if value is None or pd.isna(value):
        return ''
    value = str(value)
    value = re.sub(r'<[^>]+>', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

def simplify(text):
    text = str(text or '').lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('đ', 'd')

def parse_skill_list(value):
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    s = str(value).strip()
    if s.lower() in ('', 'nan', 'none', 'null', '[]'):
        return []
    if s.startswith('['):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip(' []\"\'') for part in re.split(r'\s*[;,\n]\s*', s) if part.strip(' []\"\'')]

def normalize_skills(value):
    skills = parse_skill_list(value)
    # matching is done on labels only, never on the model input
    search_text = ' || '.join(skills)
    normalized = simplify(search_text)
    result=[]
    for name, patterns in COMPILED_SKILLS.items():
        if any(p.search(normalized) for p in patterns):
            result.append(name)
    return result

print('Nhãn kỹ năng được chuẩn hóa:', len(SKILL_PATTERNS))
print('Ví dụ:', normalize_skills("['Lập trình Python', 'Docker', 'Cơ sở dữ liệu SQL']"))

Nhãn kỹ năng được chuẩn hóa: 48
Ví dụ: ['Python', 'SQL', 'Docker']


## 5. Lọc JD IT, loại trùng và kiểm tra phân bố nhãn
Những dòng không có bất kỳ nhãn kỹ năng chuẩn hóa nào bị loại khỏi bài toán *supervised* để tránh coi nhầm nhãn trống là âm tính. Các dòng này vẫn có thể dùng làm tài liệu cho RAG nếu muốn.

In [9]:
def category_is_it(value):
    s = simplify(value).replace(' ', '_')
    return bool(re.search(r'cong_nghe_thong_tin|cong_nghe|phan_mem|technology|information_technology', s))

def title_is_it(value):
    s = simplify(value)
    return bool(re.search(r'\b(?:developer|devops|data engineer|data scientist|software|backend|back.?end|front.?end|full.?stack|qa engineer|it support|machine learning|ai engineer|tester|sysadmin|system administrator|database administrator)\b|\blap trinh\b|\bky su phan mem\b|\bcong nghe thong tin\b|\bkiem thu phan mem\b', s))

df = df_raw.copy()
for c in ['job_title', 'description', 'requirements_text', 'technical_skills', 'category']:
    df[c] = df[c].fillna('')

print('Phân bố nhóm ngành (10 nhóm đầu):')
display(df['category'].value_counts().head(10).to_frame('jobs'))

if ONLY_IT:
    mask = df['category'].apply(category_is_it) | df['job_title'].apply(title_is_it)
    df = df.loc[mask].copy()
    print('Số JD được nhận diện là CNTT:', len(df))
    if len(df) < 100:
        print('CẢNH BÁO: Quá ít JD IT. Hãy xem tên category và/hoặc đặt ONLY_IT=False rồi chạy lại.')

# Dùng tiêu đề + mô tả + yêu cầu, KHÔNG đưa nhãn technical_skills vào X
for c in ['job_title','description','requirements_text']:
    df[c] = df[c].map(clean_text)
df['text'] = df['job_title'] + '. ' + df['description'] + ' ' + df['requirements_text']
df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
df = df.loc[df['text'].str.len() >= MIN_TEXT_CHARS].copy()

df['labels_all'] = df['technical_skills'].map(normalize_skills)
df['raw_label_present'] = df['technical_skills'].map(lambda x: len(parse_skill_list(x)) > 0)
print('JD có skills thô:', int(df['raw_label_present'].sum()))
print('JD có ít nhất một nhãn đã chuẩn hóa:', int(df['labels_all'].map(bool).sum()))

df = df.loc[df['labels_all'].map(bool)].copy()
# Loại văn bản trùng trước khi chia train/test để giảm rò rỉ.
df['text_key'] = df['text'].map(lambda x: re.sub(r'\s+', ' ', simplify(x)).strip())
df = df.drop_duplicates('text_key').reset_index(drop=True)

if MAX_ROWS is not None and len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)

print('Số JD hợp lệ để train:', len(df))
if len(df) < 100:
    raise ValueError('Số JD hợp lệ <100. Đặt ONLY_IT=False, giảm MIN_TEXT_CHARS hoặc xem dữ liệu nhãn.')
counts = Counter(label for labels in df['labels_all'] for label in labels)
display(pd.DataFrame(counts.most_common(30), columns=['skill', 'jobs']))

Phân bố nhóm ngành (10 nhóm đầu):


,jobs
category,
kinh_doanh_bán_hàng_chăm_sóc_khách_hàng,8276
sản_xuất_lao_động_phổ_thông_cơ_khí,6407
marketing_truyền_thông_quảng_cáo_nội_dung,5950
tài_chính_kế_toán_ngân_hàng_bảo_hiểm,5530
du_lịch_nhà_hàng_khách_sạn_dịch_vụ,4243
thiết_kế_nghệ_thuật_giải_trí_truyền_hình_báo_chí,3465
nhân_sự_hành_chính_pháp_chế_tư_vấn,3276
xây_dựng_kiến_trúc_bất_động_sản,2811
công_nghệ_thông_tin_kỹ_thuật_số,1906


Số JD được nhận diện là CNTT: 3097
JD có skills thô: 2736
JD có ít nhất một nhãn đã chuẩn hóa: 1162
Số JD hợp lệ để train: 1116


,skill,jobs
0,SQL,308
1,Python,275
2,Java,253
3,Git,185
4,Excel,182
5,JavaScript,180
6,AWS,121
7,MySQL,119
8,Docker,119
9,React,107


## 6. Chia dữ liệu trước khi chọn top nhãn (80/10/10)
Tập test sẽ giữ nguyên đến cuối. **Danh sách nhãn và ngưỡng chỉ xác định bằng Train/Validation**, không nhìn test.

In [10]:
train_df, holdout_df = train_test_split(df, test_size=0.20, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(holdout_df, test_size=0.50, random_state=SEED, shuffle=True)

train_counts = Counter(l for labs in train_df['labels_all'] for l in labs)
selected_labels = [label for label, ct in train_counts.most_common()
                   if ct >= MIN_TRAIN_POS][:MAX_LABELS]
if len(selected_labels) < 2:
    raise ValueError('Ít hơn 2 kỹ năng đủ mẫu. Hãy giảm MIN_TRAIN_POS hoặc đặt ONLY_IT=False.')
mlb = MultiLabelBinarizer(classes=sorted(selected_labels))
mlb.fit([[]])

def encode(y):
    return mlb.transform([[x for x in labels if x in selected_labels] for labels in y])

X_train = train_df['text'].tolist(); Y_train = encode(train_df['labels_all'])
X_val   = val_df['text'].tolist();   Y_val   = encode(val_df['labels_all'])
X_test  = test_df['text'].tolist();  Y_test  = encode(test_df['labels_all'])
print('Train / Validation / Test:', len(X_train), len(X_val), len(X_test))
print('Số lớp:', len(mlb.classes_))
print('Nhãn:', mlb.classes_.tolist())
print('Nhãn dương trong Train:', Y_train.sum(axis=0).tolist())

Train / Validation / Test: 892 112 112
Số lớp: 35
Nhãn: ['.NET', 'AWS', 'Angular', 'Azure', 'C#', 'C++', 'CI/CD', 'Docker', 'Excel', 'Figma', 'GCP', 'Git', 'HTML/CSS', 'Java', 'JavaScript', 'Jira', 'Kafka', 'Kubernetes', 'Linux', 'Machine Learning', 'Microservices', 'MongoDB', 'MySQL', 'Node.js', 'PHP', 'PostgreSQL', 'Python', 'REST API', 'React', 'Redis', 'SQL', 'Scrum', 'Spring Boot', 'TypeScript', 'Vue']
Nhãn dương trong Train: [41, 103, 40, 52, 70, 46, 55, 98, 140, 81, 39, 150, 77, 208, 138, 60, 38, 79, 74, 25, 36, 70, 99, 52, 46, 88, 226, 50, 87, 39, 246, 26, 47, 39, 26]


## 7. Huấn luyện TF-IDF + Logistic Regression (đa nhãn)
Baseline đơn giản, dễ chạy trên Colab CPU. Không phải fine-tune PhoBERT/BERT. Với mỗi skill, huấn luyện một bộ phân loại nhị phân (*One-vs-Rest*).

In [11]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=60000,
    sublinear_tf=True,
    dtype=np.float32
)
Xtr = vectorizer.fit_transform(X_train)
Xva = vectorizer.transform(X_val)
Xte = vectorizer.transform(X_test)

classifier = OneVsRestClassifier(LogisticRegression(
    solver='liblinear', max_iter=1000, class_weight='balanced', random_state=SEED
), n_jobs=1)
classifier.fit(Xtr, Y_train)
print('Đã train xong:', Xtr.shape, '| số nhãn:', Y_train.shape[1])

Đã train xong: (892, 20382) | số nhãn: 35


## 8. Chọn ngưỡng trên Validation rồi đánh giá Test
Vì đây là bài toán đa nhãn, không nên chỉ dùng accuracy. Tối ưu ngưỡng dự đoán trên Validation theo Micro F1; sau đó đánh giá một lần trên Test.

In [12]:
val_prob = classifier.predict_proba(Xva)
thresholds = np.arange(0.15, 0.71, 0.05)
results = []
for t in thresholds:
    pr = (val_prob >= t).astype(int)
    score = f1_score(Y_val, pr, average='micro', zero_division=0)
    results.append((round(float(t), 2), float(score)))
threshold, best_val_f1 = max(results, key=lambda x: x[1])
print('Best threshold (validation):', threshold)
print('Micro F1 (validation):', round(best_val_f1, 4))

test_prob = classifier.predict_proba(Xte)
test_pred = (test_prob >= threshold).astype(int)
scores = {
    'test_micro_precision': precision_score(Y_test, test_pred, average='micro', zero_division=0),
    'test_micro_recall': recall_score(Y_test, test_pred, average='micro', zero_division=0),
    'test_micro_f1': f1_score(Y_test, test_pred, average='micro', zero_division=0),
    'test_macro_f1': f1_score(Y_test, test_pred, average='macro', zero_division=0),
    'test_hamming_loss': hamming_loss(Y_test, test_pred),
}
print(json.dumps(scores, ensure_ascii=False, indent=2))
print(classification_report(Y_test, test_pred, target_names=mlb.classes_, zero_division=0))

Best threshold (validation): 0.45
Micro F1 (validation): 0.6606
{
  "test_micro_precision": 0.5496688741721855,
  "test_micro_recall": 0.7854889589905363,
  "test_micro_f1": 0.6467532467532467,
  "test_macro_f1": 0.5964370203360052,
  "test_hamming_loss": 0.06938775510204082
}
                  precision    recall  f1-score   support

            .NET       0.80      0.44      0.57         9
             AWS       0.40      1.00      0.57         8
         Angular       0.33      0.67      0.44         3
           Azure       0.40      1.00      0.57         4
              C#       0.56      0.71      0.62         7
             C++       1.00      0.62      0.77         8
           CI/CD       0.64      0.88      0.74         8
          Docker       0.55      0.79      0.65        14
           Excel       0.86      0.82      0.84        22
           Figma       0.59      0.91      0.71        11
             GCP       0.40      0.80      0.53         5
             Git       0.

## 9. Lưu model, kết quả, và dự đoán JD mới
Các file được lưu vào `MODEL_DIR`. Có thể download `.joblib`, `metrics.json` và `skill_id_mapping.csv` cho nhóm tích hợp API/Supervisor Agent.

In [13]:
skill_mapping = pd.DataFrame({
    'skill_id': [f'SK{i+1:03d}' for i in range(len(mlb.classes_))],
    'skill_name': mlb.classes_.tolist()
})
skill_mapping.to_csv(MODEL_DIR/'skill_id_mapping.csv', index=False, encoding='utf-8-sig')

artifact = {
    'vectorizer': vectorizer,
    'classifier': classifier,
    'classes': mlb.classes_.tolist(),
    'threshold': threshold,
    'skill_mapping': dict(zip(skill_mapping['skill_name'],skill_mapping['skill_id'])),
    'dataset': 'dinhieufam/VietJobs',
    'input_fields': ['job_title','description','requirements_text'],
    'label_field': 'technical_skills',
    'metrics_scope': 'weak/normalized dataset labels, not manually validated'
}
joblib.dump(artifact, MODEL_DIR/'vietjobs_skill_extractor.joblib', compress=3)
report = {'dataset':artifact['dataset'], 'n_train':len(train_df),'n_val':len(val_df),
          'n_test':len(test_df), 'n_classes':len(mlb.classes_),
          'threshold':threshold, 'val_micro_f1':best_val_f1, **scores}
(MODEL_DIR/'metrics.json').write_text(json.dumps(report, indent=2, ensure_ascii=False),encoding='utf-8')
print('Đã lưu:')
for item in MODEL_DIR.iterdir(): print(item)

def predict_skills(jd, top_k=12, min_prob=None):
    min_prob = threshold if min_prob is None else min_prob
    probs = classifier.predict_proba(vectorizer.transform([jd]))[0]
    order = np.argsort(probs)[::-1][:top_k]
    result = []
    for i in order:
        if probs[i] >= min_prob:
            label = mlb.classes_[i]
            result.append({'skill_id':artifact['skill_mapping'][label],
                           'skill':label,'confidence':round(float(probs[i]),3)})
    return result

jd_demo = 'Tuyển Python Backend Developer. Yêu cầu Python, FastAPI, SQL, Docker, Git, AWS và REST API.'
print('Ví dụ dự đoán:', predict_skills(jd_demo, min_prob=0.25))

Đã lưu:
/content/vinh_vietjobs_models/vietjobs_skill_extractor.joblib
/content/vinh_vietjobs_models/skill_id_mapping.csv
/content/vinh_vietjobs_models/metrics.json
Ví dụ dự đoán: [{'skill_id': 'SK012', 'skill': 'Git', 'confidence': 0.716}, {'skill_id': 'SK008', 'skill': 'Docker', 'confidence': 0.672}, {'skill_id': 'SK027', 'skill': 'Python', 'confidence': 0.642}, {'skill_id': 'SK028', 'skill': 'REST API', 'confidence': 0.57}, {'skill_id': 'SK026', 'skill': 'PostgreSQL', 'confidence': 0.546}, {'skill_id': 'SK015', 'skill': 'JavaScript', 'confidence': 0.488}, {'skill_id': 'SK002', 'skill': 'AWS', 'confidence': 0.48}, {'skill_id': 'SK033', 'skill': 'Spring Boot', 'confidence': 0.479}, {'skill_id': 'SK014', 'skill': 'Java', 'confidence': 0.479}, {'skill_id': 'SK018', 'skill': 'Kubernetes', 'confidence': 0.478}, {'skill_id': 'SK031', 'skill': 'SQL', 'confidence': 0.456}, {'skill_id': 'SK024', 'skill': 'Node.js', 'confidence': 0.432}]


In [14]:
# Tải toàn bộ kết quả train về máy dưới dạng ZIP
import shutil
zip_file = shutil.make_archive(str(MODEL_DIR), 'zip', root_dir=str(MODEL_DIR))
print('Tệp:', zip_file)
if Path('/content').exists():
    from google.colab import files
    files.download(zip_file)

Tệp: /content/vinh_vietjobs_models.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. (TÙY CHỌN) Embeddings + ChromaDB cho RAG
Đây là **retrieval** (truy xuất văn bản có liên quan) để làm nền cho RAG Agent, **chưa** có LLM tự sinh câu trả lời. Chỉ chạy nếu cần phần Vector Database; lần đầu sẽ cần tải mô hình embedding và xử lý tối đa `RAG_LIMIT` JD.

In [15]:
RUN_RAG = False  # Đổi thành True để chạy phần này
RAG_LIMIT = 600  # Có thể nâng lên sau khi chạy thử
if RUN_RAG:
    %pip -q install -U sentence-transformers chromadb

In [16]:
if RUN_RAG:
    import chromadb
    from sentence_transformers import SentenceTransformer
    docs = df.reset_index(drop=True).head(RAG_LIMIT).copy()
    embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    embeddings = embedder.encode(docs['text'].tolist(), batch_size=32, show_progress_bar=True,
                                  normalize_embeddings=True).astype(float).tolist()
    chroma_path = str(MODEL_DIR/'chroma_db')
    client = chromadb.PersistentClient(path=chroma_path)
    collection = client.get_or_create_collection(name='vietjobs_jds', metadata={'hnsw:space':'cosine'})
    for i in range(0,len(docs),100):
        b = docs.iloc[i:i+100]
        collection.upsert(
            ids=[f'vietjobs_{int(k)}' for k in b.index],
            documents=b['text'].tolist(),
            embeddings=embeddings[i:i+len(b)],
            metadatas=[{'title':str(t), 'category':str(c)} for t,c in zip(b['job_title'], b['category'])]
        )
    q='Việc làm Python backend và Docker tại Việt Nam'
    query_vec = embedder.encode([q], normalize_embeddings=True).astype(float).tolist()
    hits = collection.query(query_embeddings=query_vec, n_results=5,
                            include=['metadatas','distances','documents'])
    for title, distance in zip(hits['metadatas'][0], hits['distances'][0]):
        print(title['title'], '- cosine distance:', round(float(distance),3))
else:
    print('Đã bỏ qua RAG. Đổi RUN_RAG = True và chạy lại mục 10 để tạo vector DB.')

Đã bỏ qua RAG. Đổi RUN_RAG = True và chạy lại mục 10 để tạo vector DB.


## 11. Lưu ý khi viết báo cáo đồ án

1. **Dữ liệu thật:** VietJobs gồm 48.092 tin tuyển dụng thuộc 16 ngành; số mẫu dùng để train được notebook in ra sau khi lọc JD IT, nhãn trống và trùng lặp. Không được ghi 48.092 là số JD IT hay số mẫu train nếu chưa lọc.
2. **Nhãn:** `technical_skills` có sẵn trong dữ liệu, được chuẩn hóa qua bảng regex; đây không phải nhãn do nhóm gán thủ công. Các F1 chỉ phản ánh khả năng dự đoán **nhãn tham chiếu chuẩn hóa**.
3. **Đánh giá:** Train/Validation/Test tách riêng; chọn threshold chỉ trên Validation. Để báo cáo khoa học chặt chẽ, hãy gán nhãn thủ công một tập JD độc lập trước khi kết luận tính chính xác.
4. **Mở rộng:** Fine-tune PhoBERT / multilingual BERT sau khi có nhãn được duyệt và đủ GPU. RAG sẽ cần thành phần LLM nếu muốn trả lời bằng ngôn ngữ tự nhiên.

**Nguồn:** [Hugging Face VietJobs](https://huggingface.co/datasets/dinhieufam/VietJobs) · [VinNLP/VietJobs GitHub](https://github.com/VinNLP/VietJobs) · [Bài báo LREC 2026](https://aclanthology.org/2026.lrec-1.501/)